<a href="https://colab.research.google.com/github/Bri636/agents/blob/ml-bio/examples/generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!sudo apt update
!sudo apt install -y libhdf5-dev libhdf5-serial-dev hdf5-tools

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,311 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,667 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,606 kB]
Get:13 http://archive.ubuntu.com/ubunt

In [1]:
# NOTE: You may need to run this twice due to a pip dependency conflict
!pip install pip==23.3.1
!pip install git+https://github.com/ramanathanlab/genslm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/ramanathanlab/genslm to /tmp/pip-req-build-eelks8cn
  Running command git clone --filter=blob:none --quiet https://github.com/ramanathanlab/genslm /tmp/pip-req-build-eelks8cn
  Resolved https://github.com/ramanathanlab/genslm to commit e4fbf3b8e641150d708c18e12d551de8ed0cae1c
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/maxzvyagin/transformers to /tmp/pip-install-b05syw1p/transformers_bca1708fe209432dba38c99e7fb9cbb7
  Running command git clone --filter=blob:none --quiet https://github.com/maxzvyagin/transformers /tmp/pip-install-b05syw1p/transformers_bca1708fe209432dba38c99e7fb9cbb7
  Resolved https://github.com/maxzvyagin/transformers to commit ffd5aba0ad41a1ebd1897a77f6a3782fc2d75e1f
  Installing build dependencie

In [ ]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [ ]:
!ls gdrive/MyDrive/patric_250m_epoch00_val_loss_0.48_attention_removed.pt
# This currently requires you to download the 25M model weights from Globus

gdrive/MyDrive/patric_25m_epoch01-val_loss_0.57_bias_removed.pt


In [ ]:
import torch
from genslm import GenSLM

# Load model
model = GenSLM("genslm_25M_patric", model_cache_dir="/content/gdrive/MyDrive")
model.eval()

# Select GPU device if it is available, else use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Prompt the language model with a start codon
prompt = model.tokenizer.encode("ATG", return_tensors="pt").to(device)

tokens = model.model.generate(
    prompt,
    max_length=10,  # Increase this to generate longer sequences
    min_length=10,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    num_return_sequences=2,  # Change the number of sequences to generate
    remove_invalid_values=True,
    use_cache=True,
    pad_token_id=model.tokenizer.encode("[PAD]")[0],
    temperature=1.0,
)

sequences = model.tokenizer.batch_decode(tokens, skip_special_tokens=True)

for sequence in sequences:
    print(sequence)

ATG CCT TAT GGT CAC TTG TGC TCC CAC GCT
ATG TCC GGC CAA GAC AAC CAG CAC TCG GTA
